# 재현성 평가(릴리, 앙상블, 멀티)

In [1]:
import os
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. 경로 및 설정 (루프를 위한 딕셔너리)
# ==========================================
# GT 파일 경로
GT_PATH = r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\260128_Gemini-3-pro-hyper_parameter_test_채점결과\260130_GT.csv"

# 3개 모델의 Answer(리샘플링) 폴더 경로
MODELS_INFO = {
    "relevance-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_relevance_리샘플링",
    "ensemble-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_ensemble_리샘플링",
    "multiagent-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_multiagent_리샘플링"
}

# 최종 TF 채점 결과를 저장할 출력 폴더 (폴더가 없으면 생성)
OUTPUT_DIR = r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\TF_OUTPUT"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. 전처리 함수 및 기준 변수 선언 (원본 유지)
# ==========================================
def text_processor(df): 
    # ## 'not found' 값 처리
    # df = df.replace("not found", np.nan)

    def extract_loading_density(x):
        if isinstance(x, str):
            # 공백으로 잘랐을 때 첫 번째 요소가 숫자 형태인 경우 (예: "2.5 mg/cm2")
            first_part = x.split()[0]
            try:
                # 숫자만 남기기 위해 정규표현식을 쓰거나, 단위 앞부분만 float로 변환
                import re
                numeric_part = re.findall(r"[-+]?\d*\.\d+|\d+", x)
                return float(numeric_part[0]) if numeric_part else None
            except (ValueError, IndexError):
                return None
        return x

    if "Loading density (mass loading of NCM)" in df.columns:
        df["Loading density (mass loading of NCM)"] = df["Loading density (mass loading of NCM)"].apply(extract_loading_density)

    ## `Electrolyte concentration` 변수 처리
    def safe_electrolyte_convert(x):
        if type(x) == str and ('M' in x or 'm' in x):
            try:
                # 기존 로직 100% 동일하게 유지
                return float(x[:-1])
            except ValueError:
                # '1 mol L-' 처럼 숫자로 안 바뀌는 값이 나오면, 에러 내지 말고 원래 텍스트 그대로 반환
                return x
        return x

    df["Electrolyte concentration"] = df["Electrolyte concentration"].apply(safe_electrolyte_convert)
        
    # 1. 일차적으로 단위 제거 및 기호 통일
    df["Voltage range"] = df["Voltage range"].apply(
        lambda x: x.replace("V", "").replace("v", "").replace("?", "-").replace("--", "-") if pd.notna(x) else x
    )
    df["Voltage range min"] = df["Voltage range"].apply(
        lambda x: float(x.split('-')[0]) if isinstance(x, str) and '-' in x else x
    )
    df["Voltage range max"] = df["Voltage range"].apply(
        lambda x: float(x.split('-')[1]) if isinstance(x, str) and '-' in x else x
    )
    df = df.drop(columns=["Voltage range"], errors='ignore') # 에러 방지용 errors='ignore' 추가

    ## `Temperature` 변수 처리
    df["Temperature"] = df["Temperature"].apply(lambda x: 25 if x == "room temperature" else x)
        
    return df   

def c_rate_func(x): 
    if "C-rate" in x: 
        return True 
    else: 
        return False 

# 전역(Global) 설정 리스트들
Morphological_Properties_col = [
    'Particle size', 
    'Particle shape', 
    'Particle distribution', 
    'Coating layer characteristics', 
    'Crystal structure and lattice characteristics'
]

sort_col = [
    'Paper ID', 'Sample', 'Li ratio', 'Ni ratio', 'Co ratio', 'Mn ratio', 'O ratio', 'W ratio',
    'Commercial NCM used', 'Lithium source', 'Synthesis method', 'Crystallization method',
    'Crystallization final temperature', 'Crystallization final duration (hours)', 'Doping', 'Coating',
    'Active material to Conductive additive to Binder ratio', 'Electrolyte salt', 'Electrolyte concentration',
    'Electrolyte solvent', 'Electrolyte solvent ratio', 'Additive', 'Loading density (mass loading of NCM)',
    'Voltage range min', 'Voltage range max', 'Temperature', 'C-rate 0.05', 'C-rate 0.1', 
    'C-rate 0.2', 'C-rate 0.5', 'C-rate 1.0', 'C-rate 2.0', 'C-rate 4.0', 'C-rate 5.0',
    'C-rate 6.0', 'C-rate 10.0', 'C-rate 20.0', 'C-rate 40.0'
]

tf_col = [
    "Li ratio", "Ni ratio", "Co ratio", "Mn ratio", "O ratio", "W ratio", "Commercial NCM used",
    'Lithium source', "Crystallization final temperature", "Crystallization final duration (hours)", 
    "Electrolyte salt", "Electrolyte concentration", 'Loading density (mass loading of NCM)',
    'Voltage range min', 'Voltage range max', 'Temperature', 'C-rate 0.05', 'C-rate 0.1', 
    'C-rate 0.2', 'C-rate 0.5', 'C-rate 1.0', 'C-rate 2.0', 'C-rate 4.0', 'C-rate 5.0',
    'C-rate 6.0', 'C-rate 10.0', 'C-rate 20.0', 'C-rate 40.0'
]

def is_number(val):
    try:
        float(val)
        return True
    except:
        return False

# ==========================================
# 3. 데이터 로드 및 전처리 (GT는 루프 밖에서 한 번만)
# ==========================================
print("Loading Ground Truth data...")
try:
    gt_raw = pd.read_csv(GT_PATH, encoding="cp949")
except UnicodeDecodeError:
    gt_raw = pd.read_csv(GT_PATH, encoding="euc-kr")

gt = text_processor(gt_raw.copy())
gt = gt.drop(columns=[c for c in Morphological_Properties_col if c in gt.columns])
gt_c_rate = gt.columns[gt.columns.map(c_rate_func)].to_list()


# ==========================================
# 4. 메인 실행 루프 (3개 모델 x 10회차)
# ==========================================
print("\n=== TF 평가 시작 ===")

for model_name, ans_dir in MODELS_INFO.items():
    for i in range(1, 11):  # 1부터 10까지
        print(f"Processing TF: {model_name} | Iteration: {i} ...")
        
        # 파일 경로 구성
        ans_filename = f"{model_name}_{i}_resampling.xlsx"
        ans_path = os.path.join(ans_dir, ans_filename)
        
        if not os.path.exists(ans_path):
            print(f"  -> Skipping: File not found ({ans_path})")
            continue

        # LLM Answer 로드 (엑셀 파일)
        answer_raw = pd.read_excel(ans_path)
        
        # Answer 전처리
        answer = text_processor(answer_raw.copy())
        answer = answer.drop(columns=[c for c in Morphological_Properties_col if c in answer.columns])
        answer_c_rate = answer.columns[answer.columns.map(c_rate_func)].to_list()

        # key column 정의 및 비교 컬럼 설정
        key_cols = ["Paper ID", "Sample"]
        all_cols = set(gt.columns) | set(answer.columns)
        compare_cols = [col for col in all_cols if col not in key_cols]

        results = []
        all_papers = sorted(set(gt["Paper ID"]) | set(answer["Paper ID"]))

        # ==========================================
        # 5. 핵심 평가 로직 (원본 유지)
        # ==========================================
        for pid in all_papers:
            gt_group = gt[gt["Paper ID"] == pid].reset_index(drop=True)
            ans_group = answer[answer["Paper ID"] == pid].reset_index(drop=True)

            # [방어 코드] 혹시 리샘플링 개수가 다를 경우 최소 길이로 맞춤
            min_len = min(len(gt_group), len(ans_group))
            
            for idx in range(min_len):
                row_gt = gt_group.iloc[idx]
                row_ans = ans_group.iloc[idx]

                result_row = {
                    "Paper ID": pid,
                    "Sample": row_gt["Sample"]
                }

                for col in compare_cols:
                    val_gt = row_gt[col] if col in gt.columns else None
                    val_ans = row_ans[col] if col in answer.columns else None

                    try:
                        # 1. C-rate 예외 처리: 둘 다 NaN이면 비교 제외(NaN)
                        if "C-rate" in col and pd.isna(val_gt) and pd.isna(val_ans):
                            match = float('nan')
                        # 2. 둘 다 NaN이면 일치(True)
                        elif pd.isna(val_gt) and pd.isna(val_ans):
                            match = True
                        # 3. 둘 다 값이 있을 때 비교
                        elif pd.notna(val_gt) and pd.notna(val_ans):
                            if is_number(val_gt) and is_number(val_ans):
                                # 수치 데이터 오차 범위 비교
                                match = abs(float(val_gt) - float(val_ans)) <= 0.001
                            else:
                                # 텍스트 데이터 일치 비교
                                match = str(val_gt).strip() == str(val_ans).strip()
                        # 4. 한쪽만 값이 있는 경우 (누락 또는 오검출)
                        else:
                            match = False
                    except:
                        match = False

                    # 결과 저장
                    if col in tf_col:
                        result_row[col] = match
                    else:
                        result_row[col] = val_ans

                results.append(result_row)

        if not results:
            print(f"  -> No data processed for {model_name}_{i}")
            continue

        # ==========================================
        # 6. 결과 저장
        # ==========================================
        df_result = pd.DataFrame(results)

        # 누락된 열 채우기
        for col in tf_col:
            if col not in df_result.columns:
                df_result[col] = False

        # 존재하는 컬럼만 정렬에 사용 (에러 방지)
        final_sort_col = [c for c in sort_col if c in df_result.columns]
        df_result = df_result[final_sort_col]

        # 저장 파일명: acc_모델명_회차.csv
        save_name = f"acc_{model_name}_{i}.xlsx"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        
        # utf-8-sig로 저장해야 한글이 깨지지 않음
        df_result.to_excel(save_path, index=False)
        print(f"  -> Done! Saved to: {save_name}")

print("\n=== 모든 TF 평가 완료! ===")

Loading Ground Truth data...

=== TF 평가 시작 ===
Processing TF: relevance-rag | Iteration: 1 ...
  -> Done! Saved to: acc_relevance-rag_1.xlsx
Processing TF: relevance-rag | Iteration: 2 ...
  -> Done! Saved to: acc_relevance-rag_2.xlsx
Processing TF: relevance-rag | Iteration: 3 ...
  -> Done! Saved to: acc_relevance-rag_3.xlsx
Processing TF: relevance-rag | Iteration: 4 ...
  -> Done! Saved to: acc_relevance-rag_4.xlsx
Processing TF: relevance-rag | Iteration: 5 ...
  -> Done! Saved to: acc_relevance-rag_5.xlsx
Processing TF: relevance-rag | Iteration: 6 ...
  -> Done! Saved to: acc_relevance-rag_6.xlsx
Processing TF: relevance-rag | Iteration: 7 ...
  -> Done! Saved to: acc_relevance-rag_7.xlsx
Processing TF: relevance-rag | Iteration: 8 ...
  -> Done! Saved to: acc_relevance-rag_8.xlsx
Processing TF: relevance-rag | Iteration: 9 ...
  -> Done! Saved to: acc_relevance-rag_9.xlsx
Processing TF: relevance-rag | Iteration: 10 ...
  -> Done! Saved to: acc_relevance-rag_10.xlsx
Processing 

# 하이퍼파라미터 평가(27개 조합)

In [534]:
import pandas as pd
import numpy as np
import re

In [535]:
hyper_method_name = "relevance-rag(2048_100_9)_resampling"

gt = pd.read_csv("C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/260130_GT.csv", encoding="cp949")
answer = pd.read_csv(f"C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/리샘플링_CSV/{hyper_method_name}.csv", encoding="cp949") ## .drop(columns="Additional treatment")

In [536]:
# gt와 answer의 'Paper ID'와 'Sample' 컬럼을 기준으로 정렬하는건데 리샘플링 햇다면 이 셀 지워도 될 듯
#gt = gt.sort_values(by=['Paper ID', 'Sample']).reset_index(drop=True)
#answer = answer.sort_values(by=['Paper ID', 'Sample']).reset_index(drop=True)

In [537]:
# 위의 셀 실행 후 잘 정렬되엇는지 확인하는건데 리샘플링한 파일은 불필요
#gt[['Paper ID', 'Sample']] == answer[['Paper ID', 'Sample']][0:27]

In [538]:
def text_processor(df): 
    # ## 'not found' 값 처리
    # df = df.replace("not found", np.nan)

    def extract_loading_density(x):
        if isinstance(x, str):
            # 공백으로 잘랐을 때 첫 번째 요소가 숫자 형태인 경우 (예: "2.5 mg/cm2")
            first_part = x.split()[0]
            try:
                # 숫자만 남기기 위해 정규표현식을 쓰거나, 단위 앞부분만 float로 변환
                # 여기서는 단순히 첫 단어에서 숫자/소수점만 남기는 방식을 제안합니다.
                import re
                numeric_part = re.findall(r"[-+]?\d*\.\d+|\d+", x)
                return float(numeric_part[0]) if numeric_part else None
            except (ValueError, IndexError):
                return None
        return x

    if "Loading density (mass loading of NCM)" in df.columns:
        df["Loading density (mass loading of NCM)"] = df["Loading density (mass loading of NCM)"].apply(extract_loading_density)

    ## `Electrolyte concentration` 변수 처리
    df["Electrolyte concentration"] = df["Electrolyte concentration"].apply(
        lambda x: float(x[:-1]) if (type(x) == str) and ('M' in x or 'm' in x) else x
    )
        
    # 1. 일차적으로 단위 제거 및 기호 통일
    df["Voltage range"] = df["Voltage range"].apply(
        lambda x: x.replace("V", "").replace("v", "").replace("?", "-").replace("--", "-") if pd.notna(x) else x
    )
    df["Voltage range min"] = df["Voltage range"].apply(
        lambda x: float(x.split('-')[0]) if isinstance(x, str) and '-' in x else x
    )
    df["Voltage range max"] = df["Voltage range"].apply(
        lambda x: float(x.split('-')[1]) if isinstance(x, str) and '-' in x else x
    )
    df = df.drop(columns=["Voltage range"])


    ## `Temperature` 변수 처리
    df["Temperature"] = df["Temperature"].apply(lambda x: 25 if x == "room temperature" else x)
        
    return df   

In [539]:
#def combine_duplicated_cols(df):
    #int_crate_cols = [
    #    col for col in df.columns
    #    if re.fullmatch(r"C-rate\s+[0-9]+", col)
    #]    
    
    #int2float_map = {}
    #for int_col in int_crate_cols:
    #    int2float_map[f"{int_col}"] = int_col[:7] + f"{float(int_col[7:])}"

    #for int_col, float_col in int2float_map.items():
    #    if float_col in df.columns:
    #        df[int_col] = df[int_col].combine_first(df[float_col])
    #        df.drop(columns=float_col, inplace=True)
    #    df.rename(columns={int_col: float_col}, inplace=True)
    
    #return df

In [540]:
def c_rate_func(x): 
    if "C-rate" in x: 
        return True 
    else: 
        return False 

In [541]:
gt = text_processor(gt)
answer = text_processor(answer)

In [542]:
#gt = combine_duplicated_cols(gt)
#answer = combine_duplicated_cols(answer)

In [543]:
Morphological_Properties_col = [
    'Particle size', 
    'Particle shape', 
    'Particle distribution', 
    'Coating layer characteristics', 
    'Crystal structure and lattice characteristics'
]

gt = gt.drop(columns=Morphological_Properties_col)
answer = answer.drop(columns=Morphological_Properties_col)

In [544]:
gt_c_rate = gt.columns[gt.columns.map(c_rate_func)].to_list()
answer_c_rate = answer.columns[answer.columns.map(c_rate_func)].to_list()

In [545]:
# other_c_rate_col = [
#     'C-rate 0.05',
#     'C-rate 5.0',
#     'C-rate 10.0',
#     'C-rate 20.0',
#     'C-rate 40.0',
#     'C-rate 0.33',
#     'C-rate None',
#     'C-rate 0.2',
#     'C-rate 0.3',
#     'C-rate 4.0',
#     'C-rate 6.0',
# ]

In [546]:
paper_id = gt["Paper ID"].unique()

In [547]:
sort_col = [
    'Paper ID',
    'Sample',
    'Li ratio',
    'Ni ratio',
    'Co ratio',
    'Mn ratio',
    'O ratio',
    'W ratio',
    'Commercial NCM used',
    'Lithium source',
    'Synthesis method',
    'Crystallization method',
    'Crystallization final temperature',
    'Crystallization final duration (hours)',
    'Doping',
    'Coating',
    'Active material to Conductive additive to Binder ratio',
    'Electrolyte salt',
    'Electrolyte concentration',
    'Electrolyte solvent',
    'Electrolyte solvent ratio',
    'Additive',
    'Loading density (mass loading of NCM)',
    'Voltage range min',
    'Voltage range max',
    'Temperature',
    'C-rate 0.05',
    'C-rate 0.1', 
    'C-rate 0.2', 
    'C-rate 0.5', 
    'C-rate 1.0', 
    'C-rate 2.0',
    'C-rate 4.0',
    'C-rate 5.0',
    'C-rate 6.0', 
    'C-rate 10.0', 
    'C-rate 20.0', 
    'C-rate 40.0'
    # 'C-rate None', 
]

In [548]:
tf_col = [
    "Li ratio", 
    "Ni ratio", 
    "Co ratio", 
    "Mn ratio", 
    "O ratio", 
    "W ratio", 
    "Commercial NCM used",
    'Lithium source',
    "Crystallization final temperature", 
    "Crystallization final duration (hours)", 
    "Electrolyte salt", 
    "Electrolyte concentration",
    'Loading density (mass loading of NCM)',
    'Voltage range min',
    'Voltage range max',
    'Temperature',
    'C-rate 0.05',
    'C-rate 0.1', 
    'C-rate 0.2', 
    'C-rate 0.5', 
    'C-rate 1.0', 
    'C-rate 2.0',
    'C-rate 4.0',
    'C-rate 5.0',
    'C-rate 6.0', 
    'C-rate 10.0', 
    'C-rate 20.0', 
    'C-rate 40.0'
    # 'C-rate None', 

]

In [549]:
def is_number(val):
    try:
        float(val)
        return True
    except:
        return False

# key column 정의
key_cols = ["Paper ID", "Sample"]

# 모든 열 모음 (비교 대상)
all_cols = set(gt.columns) | set(answer.columns)
compare_cols = [col for col in all_cols if col not in key_cols]

# 결과 저장
results = []

# 모든 논문 처리
all_papers = sorted(set(gt["Paper ID"]) | set(answer["Paper ID"]))

# 모든 논문 처리 (이미 정렬과 행 개수가 맞으므로 gt를 기준으로 루프)
for pid in all_papers:
    # 해당 논문의 데이터 추출 (이미 리샘플링으로 행 개수가 일치하는 상태)
    gt_group = gt[gt["Paper ID"] == pid].reset_index(drop=True)
    ans_group = answer[answer["Paper ID"] == pid].reset_index(drop=True)

    # 행 개수가 같으므로 단순히 gt_group의 길이만큼 루프
    for i in range(len(gt_group)):
        row_gt = gt_group.iloc[i]
        row_ans = ans_group.iloc[i]

        result_row = {
            "Paper ID": pid,
            "Sample": row_gt["Sample"]
        }

        for col in compare_cols:
            val_gt = row_gt[col] if col in gt.columns else None
            val_ans = row_ans[col] if col in answer.columns else None

            try:
                # 1. C-rate 예외 처리: 둘 다 NaN이면 비교 제외(NaN)
                if "C-rate" in col and pd.isna(val_gt) and pd.isna(val_ans):
                    match = float('nan')
                # 2. 둘 다 NaN이면 일치(True)
                elif pd.isna(val_gt) and pd.isna(val_ans):
                    match = True
                # 3. 둘 다 값이 있을 때 비교
                elif pd.notna(val_gt) and pd.notna(val_ans):
                    if is_number(val_gt) and is_number(val_ans):
                        # 수치 데이터 오차 범위 비교
                        match = abs(float(val_gt) - float(val_ans)) <= 0.001
                    else:
                        # 텍스트 데이터 일치 비교
                        match = str(val_gt).strip() == str(val_ans).strip()
                # 4. 한쪽만 값이 있는 경우 (누락 또는 오검출)
                else:
                    match = False
            except:
                match = False

            # 결과 저장
            if col in tf_col:
                result_row[col] = match
            else:
                result_row[col] = val_ans

        results.append(result_row)

# 결과 DataFrame
df_result = pd.DataFrame(results)

# 누락된 열 채우기 (혹시라도)
for col in tf_col:
    if col not in df_result.columns:
        df_result[col] = False

# 컬럼 순서 정렬
df_result = df_result[sort_col]

# 저장
df_result.to_csv(f"C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/TF_OUTPUT/acc_{hyper_method_name}.csv", index=False)